<html>importa pandas y crea el objeto Data frame del comma separated values  <html/>

In [1]:
import pandas as pd
import json
import sqlite3
import os

#se crea un objetos tipo D::::ATAFRAME con ayuda de pandas con su funcion read_csv
#que lee el CSV (comma separated values)
arcs_data_frame = pd.read_csv('../data/raw/arc_enemies.csv')
items_data_frame = pd.read_csv('../data/raw/items.csv')
quest_data_frame = pd.read_csv('../data/raw/quests.csv')




<html>muestra la info general de los  data frame total de filas y columnas</html>

In [2]:
print("Enemigos/Arcs (filas, columnas: ", arcs_data_frame.shape)
print("items (filas, columnas): ", items_data_frame.shape)
print("quest (filas, columnas): ", quest_data_frame.shape)


Enemigos/Arcs (filas, columnas:  (21, 9)
items (filas, columnas):  (500, 35)
quest (filas, columnas):  (98, 15)


In [3]:
#Se cuentan las filas de cuantos valores nulos (NaN) hay por cada columna en el DataFrame de enemigos (Arcs)
print("\n@@@ CONTEO DE NULOS EN LOS ARCS @@@")
print(arcs_data_frame.isna().sum())




#filtra las filas donde 'markers' es nulo para ver que enemigos tienen este problema.
#arcs_data_frame['markers'].isna() devuelve una lista de true/false (Boolean Mask)

missing_rows = arcs_data_frame[arcs_data_frame['markers'].isna()]

print("\n @@ Enemigos sin lugares de hubicacion relacionados @@")
display(missing_rows[['id', 'name', 'relatedLocationTypes', 'markers']])

print("""\nSe identifica que que los Arcs con nombre (firefly, hornet, snitch, wasp) son arcs de tipo dron patrulla dinamica, 
No es un error del dataSet, ya que estos arcs no tienen cordenadas fijas porque patrullan por todo el mapa.""")


#Revision de columas clave de items y Quest que se van a usar mas adelante

cols_items_clave = "id name rarity type value weight breaksInto".split()


print ("\n\n\npequeno print para ver todoos los items legendarios")
raresa_items = "id name rarity".split()
items_legendarios = items_data_frame.loc[items_data_frame['rarity'] == 'legendary', raresa_items]
display(items_legendarios)

print ("\n\n\n@@ CONTEO DE LOS VALORES NULOS EN LAS COLUMNAS CLAVE DE ITEMS @@")
print(items_data_frame[cols_items_clave].isna().sum())


print("podemos observar que 90 items no tienen rareza definida, y que 203 items no se pueden desarmar/reciclar")





@@@ CONTEO DE NULOS EN LOS ARCS @@@
updatedAt               0
id                      0
name                    0
dropTable               0
relatedLocationTypes    4
icon                    0
image                   0
markers                 4
relatedMaps             0
dtype: int64

 @@ Enemigos sin lugares de hubicacion relacionados @@


,id,name,relatedLocationTypes,markers
4,firefly,Firefly,NaN,NaN
5,hornet,Hornet,NaN,NaN
13,snitch,Snitch,NaN,NaN
20,wasp,Wasp,NaN,NaN



Se identifica que que los Arcs con nombre (firefly, hornet, snitch, wasp) son arcs de tipo dron patrulla dinamica, 
No es un error del dataSet, ya que estos arcs no tienen cordenadas fijas porque patrullan por todo el mapa.



pequeno print para ver todoos los items legendarios


,id,name,rarity
0,acoustic_guitar,Acoustic Guitar,legendary
18,anvil_splitter,Anvil Splitter,legendary
23,aphelion,Aphelion,legendary
137,dolabra,Dolabra,legendary
147,equalizer,Equalizer,legendary
239,jupiter,Jupiter,legendary
245,kinetic_converter,Kinetic Converter,legendary
251,leviathans_crown_ship_model,"""Leviathan's Crown"" Ship Model",legendary
274,matriarch_reactor,Matriarch Reactor,legendary
325,queen_reactor,Queen Reactor,legendary





@@ CONTEO DE LOS VALORES NULOS EN LAS COLUMNAS CLAVE DE ITEMS @@
id              0
name            0
rarity         90
type            0
value           0
weight          0
breaksInto    203
dtype: int64
podemos observar que 90 items no tienen rareza definida, y que 203 items no se pueden desarmar/reciclar


In [4]:
#se toma el texto crudo de la columna drop table, unicamente del primer enemigo (.iloc[0] = Bastion)
raw_drop_table_sample = arcs_data_frame['dropTable'].iloc[0] 

#Se imprime el tipo de dato original 
print("tipo de dato original en el csv: ", type( raw_drop_table_sample))



#Se convierte el texto json en una estructura nativa de Python usando json/loads()
parsed_drop_table_sample = json.loads(raw_drop_table_sample)
print("\nTipo de dato desdepues de json.loads(): ", type(parsed_drop_table_sample))
print("\nPrimer item que suelta el enemigo", parsed_drop_table_sample[0])

tipo de dato original en el csv:  <class 'str'>

Tipo de dato desdepues de json.loads():  <class 'list'>

Primer item que suelta el enemigo {'foundIn': ['ARC'], 'icon': '/items/icons/arc_alloy.webp', 'id': 'arc_alloy', 'name': 'ARC Alloy', 'rarity': 'uncommon', 'type': 'topside material', 'updatedAt': '2025-10-20T11:29:16.330Z', 'value': 200}


In [5]:
#se crea una lista vacia en memoria, aqui se ira depositando la fila desampacada de enemy_drop_rows

enemy_drops_rows = []
#.iterrows() nos permite recorrer las 21 filas de arcs_data_frame una por una.
# // 'index' guarda el número de fila (0 al 20) y 'row' contiene los datos de ese enemigo.

for index , row in arcs_data_frame.iterrows():
    enemy_id = row['id'] # // Extraemos el identificador 
    enemy_name = row['name'] #// y nombre del enemigo actual
    
    if pd.notna(row['dropTable']):
        drops_list = json.loads(row['dropTable'])# // Seguridad ante nulos: verificamos que 'dropTable' no sea NaN antes de usar json.loads()
        
        for item in drops_list: # // Segundo ciclo (anidado): recorremos cada diccionario 'item' dentro de la lista de ese enemigo
            enemy_drops_rows.append({
            'enemy_id': enemy_id,
            'enemy_name': enemy_name,
            'item_id': item.get('id'),
            'item_name': item.get('name'),
            'rarity': item.get('rarity', 'unknown'),
            'type': item.get('type', 'unknown'),
            'value': item.get('value', 0)
        })
        

#Se convierte la lista de diccionarios en un dataFrame tabular plano
df_enemy_drops = pd.DataFrame(enemy_drops_rows)


        #se valida el resultado 
print(f"Transformacion exitosa: de {len(arcs_data_frame)} enemigos extrajimos {len(df_enemy_drops)} registros de drops.")
display(df_enemy_drops.head(10))

    

Transformacion exitosa: de 21 enemigos extrajimos 116 registros de drops.


,enemy_id,enemy_name,item_id,item_name,rarity,type,value
0,bastion,Bastion,arc_alloy,ARC Alloy,uncommon,topside material,200
1,bastion,Bastion,advanced_arc_powercell,Advanced ARC Powercell,rare,misc,640
2,bastion,Bastion,arc_performance_steel,ARC Performance Steel,rare,recyclable,1000
3,bastion,Bastion,bastion_cell,Bastion Cell,epic,recyclable,5000
4,bastion,Bastion,arc_motion_core,ARC Motion Core,rare,topside material,1000
5,bastion,Bastion,mechanical_components,Mechanical Components,uncommon,refined material,640
6,bastion,Bastion,arc_flex_rubber,ARC Flex Rubber,rare,recyclable,1000
7,bombardier,Bombardier,launcher_ammo,Launcher Ammo,rare,ammunition,250
8,bombardier,Bombardier,advanced_arc_powercell,Advanced ARC Powercell,rare,misc,640
9,bombardier,Bombardier,arc_flex_rubber,ARC Flex Rubber,rare,recyclable,1000


In [6]:
#Lista patra la tabla puente (enemigo <-> mapa) y diccionario para no duplicar el catalogo de mapas
enemy_maps_rows = []
unique_maps_dict = {}

for index, row in arcs_data_frame.iterrows():
    enemy_id = row['id']

    if pd.notna(row['relatedMaps']):
        maps_list = json.loads(row['relatedMaps'])

        for mapa in maps_list:
            
            map_id = mapa.get('id')
            map_name = mapa.get('name')

            enemy_maps_rows.append({
                'enemy_id': enemy_id,
                'map_id': map_id
            })

             # // Guardamos el mapa en el diccionario usando 'map_id' como llave.
            # // Si el mapa 'dam' sale 15 veces, el diccionario simplemente sobrescribe la llave 'dam',
            # // garantizando que al final tengamos cada mapa UNA sola vez (sin duplicados).


            unique_maps_dict[map_id] = {
                'id': map_id,
                'name': map_name
            }
# // 3. Convertimos ambas estructuras en DataFrames limpios
# // .drop_duplicates() asegura que no haya pares (enemy_id, map_id) repetidos
df_enemy_maps = pd.DataFrame(enemy_maps_rows).drop_duplicates()
df_maps = pd.DataFrame(list(unique_maps_dict.values()))

print(f"Catalogo de mapas unicos ({len(df_maps)} mapas): ")
display(df_maps)
print(f"Total de relaciones enemigo-mapa: ({len(df_enemy_maps)}")
display(df_enemy_maps.head(81))

Catalogo de mapas unicos (6 mapas): 


,id,name
0,dam,Dam Battlegrounds
1,spaceport,The Spaceport
2,buried-city,Buried City
3,blue-gate,Blue Gate
4,stella-montis,Stella Montis
5,riven-tides,Riven Tides


Total de relaciones enemigo-mapa: (81


,enemy_id,map_id
0,bastion,dam
1,bastion,spaceport
2,bastion,buried-city
3,bastion,blue-gate
4,bastion,stella-montis
...,...,...
76,vaporizer,dam
77,vaporizer,spaceport
78,vaporizer,buried-city
79,vaporizer,blue-gate


In [7]:

#Seleccionamos solo las columnas atomicas (Planas) de items_data_frame y creamos una copia con .copy()
cols_planas_items = ['id', 'name', 'rarity', 'type', 'value', 'weight', 'stackSize']
df_items_clean = items_data_frame[cols_planas_items].copy()


#Se tratan los nulos, se rellenan los 90 nulos NaN de 'rarity' con la etiqueta 'none'
df_items_clean['rarity'] = df_items_clean['rarity'].fillna('none')

#Se extrae la tabla de reciclaje de (breakInto) para saber en que componentes se rompe cada item
recycle_rows = []

for index, row in items_data_frame.iterrows():
    if pd.notna(row['breaksInto']) and row['breaksInto'] != '[]':
        try:
            components_list = json.loads(row['breaksInto'])
            for comp in components_list:
                sub_item = comp.get('item', {})
                recycle_rows.append({
                    'parent_item_id': row['id'],
                    'parent_item_name': row['name'],
                    'parent_value': row['value'],
                    'component_id': sub_item.get('id', None),
                    'component_name': sub_item.get('name', None),
                    'amount': comp.get('amount', 0),
                    'component_unit_value': sub_item.get('value', 0)
                })
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON for row {index}: {e}")
            continue

df_item_recycling = pd.DataFrame(recycle_rows)
print("Relaciones de reciclaje extraidas: {len(df_item_recycling)}")
display(df_item_recycling.head(5))

     

Relaciones de reciclaje extraidas: {len(df_item_recycling)}

,parent_item_id,parent_item_name,parent_value,component_id,component_name,amount,component_unit_value
0,acoustic_guitar,Acoustic Guitar,7000,wires,Wires,6,200
1,acoustic_guitar,Acoustic Guitar,7000,metal_parts,Metal Parts,4,75
2,adrenaline_shot,Adrenaline Shot,300,chemicals,Chemicals,1,50
3,adrenaline_shot,Adrenaline Shot,300,plastic_parts,Plastic Parts,1,60
4,advanced_arc_powercell,Advanced ARC Powercell,640,arc_powercell,ARC Powercell,2,270


In [8]:

#Seleccionamos solo las columnas atomicas (Planas) de items_data_frame y creamos una copia con .copy()
cols_planas_items = ['id', 'name', 'rarity', 'type', 'value', 'weight', 'stackSize']
df_items_clean = items_data_frame[cols_planas_items].copy()


#Se tratan los nulos, se rellenan los 90 nulos NaN de 'rarity' con la etiqueta 'none'
df_items_clean['rarity'] = df_items_clean['rarity'].fillna('none')

#Se extrae la tabla de reciclaje de (breakInto) para saber en que componentes se rompe cada item
recycle_rows = []

for index, row in items_data_frame.iterrows():#se recuerda que hay 203 nulos en breakInto. el if pd.notna hace que el codigo no truene
    if pd.notna(row['breaksInto']) and row['breaksInto'] != '[]':
        components_list = json.loads(row['breaksInto'])

        for comp in components_list:
            
            # // Dentro de 'breaksInto', cada elemento tiene 'amount' (cantidad) y un sub-diccionario 'item'
            sub_item = comp.get('item', {})
            recycle_rows.append({
            'parent_item_id': row['id'],   # // Ítem original que vamos a romper (ej. 'acoustic_guitar')
            'parent_item_name': row['name'],  # // Nombre del ítem originaleeee
            'parent_value': row['value'],        # // Cuánto vale si lo vendemos entero (ej. 7000)
            'component_id': sub_item.get('id'), # // Material que obtenemos al romperlo (ej. 'wires')
            'component_name': sub_item.get('name'),  # // Nombre del material obtenido
            'amount': comp.get('amount', 0),  # // Cuántas unidades nos da (ej. 6)
            'component_unit_value': sub_item.get('value', 0) # // Valor unitario del material (ej. 200)
                 })
                

df_item_recycling = pd.DataFrame(recycle_rows)
print(f"Relaciones de reciclaje extraidas: {len(df_item_recycling)}")
display(df_item_recycling.head(5))


Relaciones de reciclaje extraidas: 508


,parent_item_id,parent_item_name,parent_value,component_id,component_name,amount,component_unit_value
0,acoustic_guitar,Acoustic Guitar,7000,wires,Wires,6,200
1,acoustic_guitar,Acoustic Guitar,7000,metal_parts,Metal Parts,4,75
2,adrenaline_shot,Adrenaline Shot,300,chemicals,Chemicals,1,50
3,adrenaline_shot,Adrenaline Shot,300,plastic_parts,Plastic Parts,1,60
4,advanced_arc_powercell,Advanced ARC Powercell,640,arc_powercell,ARC Powercell,2,270
